# Quantum Phase Estimation

## Given a unitary $U$ and an eigenstate $|u\rangle$ with $U|u\rangle = e^{2\pi i\theta}|u\rangle$, Quantum Phase Estimation (QPE) returns an $n$-bit approximation of $\theta$. It is the workhorse subroutine behind Shor, HHL, and almost any algorithm that involves eigenvalues.

# 1. Problem statement

## Inputs:

## - A unitary $U$ with eigenvector $|u\rangle$ and eigenvalue $e^{2\pi i\theta}$, with $\theta \in [0, 1)$.
## - The ability to apply controlled-$U^{2^k}$ for $k = 0, 1, \dots, n-1$.

## Output: an $n$-bit estimate of $\theta$, accurate to $\pm 2^{-n}$.

# 2. The algorithm

## 1. Prepare $n$ "counting" qubits in $|0\rangle^{\otimes n}$ and the eigenstate $|u\rangle$ in a second register.
## 2. Apply $H^{\otimes n}$ to the counting register.
## 3. For each counting qubit $k$ (from least significant), apply controlled-$U^{2^k}$ to the second register.
## 4. Apply the inverse QFT to the counting register.
## 5. Measure the counting register. The bitstring read out is $\theta \cdot 2^n$ (rounded).

# 3. Why it works

## After step 3 the counting register holds

$$ \Large  \frac{1}{\sqrt{2^n}} \sum_{k=0}^{2^n-1} e^{2\pi i\, k\theta} |k\rangle. $$

## This is exactly the QFT of $|\theta \cdot 2^n\rangle$ (assuming $\theta \cdot 2^n$ is an integer; otherwise the result is concentrated near the closest integer). Applying $\mathrm{QFT}^{-1}$ collapses the state to $|\theta \cdot 2^n\rangle$ — measurement reveals the bits of $\theta$.

## For non-integer $\theta \cdot 2^n$ the measurement is a probability distribution sharply peaked around the closest integer — the inaccuracy decays as $2^{-n}$.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT
import numpy as np

def qpe_circuit(theta, n_counting):
    """Estimate theta where U = phase gate exp(2 pi i theta) on |1>.
    The 'eigenstate' is |1> (a single qubit)."""
    qc = QuantumCircuit(n_counting + 1, n_counting)
    qc.x(n_counting)                                 # eigenstate |1>
    qc.h(range(n_counting))                          # superposition on counting
    for k in range(n_counting):
        angle = 2 * np.pi * theta * (2 ** k)
        qc.cp(angle, k, n_counting)                  # controlled U^{2^k}
    qc.append(QFT(n_counting, inverse=True, do_swaps=True), range(n_counting))
    qc.measure(range(n_counting), range(n_counting))
    return qc

theta = 1/3   # try to estimate
n = 6
qc = qpe_circuit(theta, n)
sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=2048).result().get_counts()
best = max(counts.items(), key=lambda kv: kv[1])[0]
estimate = int(best[::-1], 2) / 2**n            # reverse little-endian
print(f'True theta : {theta:.4f}')
print(f'Estimate  : {estimate:.4f} (from bitstring {best})')

# 4. Cost

## - $n$ counting qubits + register for $|u\rangle$.
## - $O(n)$ controlled-$U^{2^k}$ calls. The expensive one is $U^{2^{n-1}}$ — making this efficient requires fast classical   exponentiation of the underlying operation. In Shor, this is **modular exponentiation** (next notebook).
## - $O(n^2)$ gates for the inverse QFT.

# 5. Where it shows up

## - **Shor's algorithm**: phase estimation on the modular exponentiation operator extracts the period.
## - **HHL algorithm**: estimate the eigenvalues of $A$ to invert it.
## - **Quantum chemistry**: estimate ground-state energies of molecular Hamiltonians.
## - **Quantum simulation**: pick out eigenvalues of any unitary you can implement.

# Recap

## - Estimates the phase $\theta$ of $U|u\rangle = e^{2\pi i\theta}|u\rangle$ to $n$ bits.
## - Uses $n$ counting qubits, $O(n)$ controlled-$U^{2^k}$ calls, and an inverse QFT.
## - The most-used quantum subroutine after the QFT itself.